In [1]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import IsolationForest
import warnings
warnings.filterwarnings('ignore')

# 1. Автоматическое приведение типов

In [2]:
class AutoTypeConverter(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        # Удаляем ненужные столбцы
        cols_to_drop = ['Unnamed: 0', 'Accident_Index', 'LSOA_of_Accident_Location']
        for col in cols_to_drop:
            if col in X.columns:
                X = X.drop(columns=[col])
        
        # Приведение типов
        for col in X.select_dtypes(include=['object']).columns:
            # Попытка преобразовать в числовой, если возможно
            try:
                # Сохраняем NaN как NaN
                temp = pd.to_numeric(X[col], errors='coerce')
                if not temp.isna().all():  # если удалось преобразовать хотя бы часть
                    X[col] = temp
            except Exception:
                pass  # остаётся object
        
        return X

# 2. Извлечение признаков из Date и Time

In [3]:
class DateTimeFeatureExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        if 'Date' in X.columns:
            X['Date'] = pd.to_datetime(X['Date'], errors='coerce')
            X['Year'] = X['Date'].dt.year
            X['Month'] = X['Date'].dt.month
            X['Day'] = X['Date'].dt.day
            X['DayOfWeek'] = X['Date'].dt.dayofweek
            X['IsWeekend'] = X['DayOfWeek'].isin([5,6]).astype(int)
            X = X.drop(columns=['Date'])
        
        # Если уже есть Year — удаляем исходный
        if 'Year' in X.columns and X['Year'].dtype in [np.int64, np.int32]:
            # Предполагаем, что это исходный Year → удаляем, если он дублирует
            # Но безопаснее: оставить только извлечённый
            pass  # на самом деле, если исходный Year совпадает — можно оставить один
        
        if 'Time' in X.columns:
            X['Time'] = pd.to_datetime(X['Time'], format='%H:%M', errors='coerce').dt.time
            X['Hour'] = X['Time'].apply(lambda x: x.hour if pd.notnull(x) else np.nan)
            X['Minute'] = X['Time'].apply(lambda x: x.minute if pd.notnull(x) else np.nan)
            X['TimeOfDay'] = X['Hour'].apply(
                lambda h: 'Night' if pd.isna(h) or h < 6 else
                          'Morning' if h < 12 else
                          'Afternoon' if h < 18 else 'Evening'
            )
            X = X.drop(columns=['Time'])
        
        return X

# 3. Обработка геокоординат

In [4]:
class GeoCoordinateProcessor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        geo_cols = ['Longitude', 'Latitude']
        for col in geo_cols:
            if col in X.columns:
                # Заполняем NaN средним (или можно использовать KNN)
                X[col] = X[col].fillna(X[col].median())
                # Нормализуем
                X[col] = (X[col] - X[col].mean()) / X[col].std()
        return X

# 4. Обработка аномалий (IQR + замена на границы)

In [5]:
class OutlierHandler(BaseEstimator, TransformerMixin):
    def __init__(self, columns=None):
        self.columns = columns
        self.bounds_ = {}

    def fit(self, X, y=None):
        if self.columns is None:
            self.columns = X.select_dtypes(include=[np.number]).columns.tolist()
        for col in self.columns:
            Q1 = X[col].quantile(0.25)
            Q3 = X[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            self.bounds_[col] = (lower_bound, upper_bound)
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.columns:
            if col in X.columns:
                lower, upper = self.bounds_[col]
                X[col] = np.where(X[col] < lower, lower, X[col])
                X[col] = np.where(X[col] > upper, upper, X[col])
        return X

# 5. Обработка пропущенных значений

In [6]:
class MissingValueHandler(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.numeric_imputer = SimpleImputer(strategy='median')
        self.categorical_imputer = SimpleImputer(strategy='most_frequent')
        self.numeric_cols_ = None
        self.categorical_cols_ = None

    def fit(self, X, y=None):
        self.numeric_cols_ = X.select_dtypes(include=[np.number]).columns.tolist()
        self.categorical_cols_ = X.select_dtypes(include=['object']).columns.tolist()
        self.numeric_imputer.fit(X[self.numeric_cols_])
        self.categorical_imputer.fit(X[self.categorical_cols_])
        return self

    def transform(self, X):
        X = X.copy()
        if len(self.numeric_cols_) > 0:
            X[self.numeric_cols_] = self.numeric_imputer.transform(X[self.numeric_cols_])
        if len(self.categorical_cols_) > 0:
            X[self.categorical_cols_] = self.categorical_imputer.transform(X[self.categorical_cols_])
        return X

# 6. Основной пайплайн

In [7]:
def create_uk_accidents_pipeline():
    # Этапы предобработки
    pipeline = Pipeline([
        ('auto_type', AutoTypeConverter()),
        ('datetime_features', DateTimeFeatureExtractor()),
        ('geo_coords', GeoCoordinateProcessor()),
        ('outliers', OutlierHandler()),
        ('missing_values', MissingValueHandler()),
        ('final_preprocessor', ColumnTransformer(
            transformers=[
                ('num', StandardScaler(), [
                    'Location_Easting_OSGR',
                    'Location_Northing_OSGR',
                    'Longitude',
                    'Latitude',
                    'Police_Force',
                    'Accident_Severity',
                    'Number_of_Vehicles',
                    'Number_of_Casualties',
                    'Day_of_Week',
                    'Local_Authority_(District)',
                    '1st_Road_Class',
                    '1st_Road_Number',
                    'Speed_limit',
                    '2nd_Road_Class',
                    '2nd_Road_Number',
                    'Urban_or_Rural_Area',
                    'Year',      # извлечён из Date
                    'Month',
                    'Day',
                    'DayOfWeek',
                    'IsWeekend',
                    'Hour',
                    'Minute'
                ]),
                ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), [
                    'Local_Authority_(Highway)',
                    'Road_Type',
                    'Junction_Control',
                    'Pedestrian_Crossing-Human_Control',
                    'Pedestrian_Crossing-Physical_Facilities',
                    'Light_Conditions',
                    'Weather_Conditions',
                    'Road_Surface_Conditions',
                    'Special_Conditions_at_Site',
                    'Carriageway_Hazards',
                    'TimeOfDay'
                ])
            ],
            remainder='drop'  # автоматически удаляет Accident_Index, LSOA, Year (если был), Unnamed: 0 и др.
        ))
    ])
    return pipeline

# Пример использования

In [8]:
if __name__ == "__main__":
    # Загрузите ваш DataFrame (df) здесь
    df = pd.read_csv(r'datasets/UK_Accident.csv')

    # Создаем пайплайн
    pipeline = create_uk_accidents_pipeline()

    # Преобразуем данные
    df_processed = pipeline.fit_transform(df)

    print("✅ Обработка завершена!")
    print(f"Размер обработанных данных: {df_processed.shape}")

✅ Обработка завершена!
Размер обработанных данных: (200362, 285)
